## 4.2 Verificación Comparativa de Migración (MySQL vs. MongoDB)

**Requerimiento:**
Mostrar como evidencia la cantidad de documentos por colección (`count_documents`) junto a los `SELECT COUNT(*)` equivalentes sobre la base relacional MySQL, verificando la consistencia e integridad de los datos migrados sin pérdida de información.

In [ ]:
import mysql.connector
import pandas as pd

# Conexión a la base de datos origen en MySQL
mysql_conn = mysql.connector.connect(
    host="127.0.0.1",
    user="root",
    password="tu_nueva_clave",
    database="saludusm"
)
cursor = mysql_conn.cursor(dictionary=True)

def get_mysql_count(tabla):
    cursor.execute(f"SELECT COUNT(*) AS total FROM {tabla}")
    return cursor.fetchone()["total"]

# Conteo de entidades anidadas en MongoDB (atenciones y recetas dentro de citas)
atenciones_mongo = db.citas.count_documents({"atencion": {"$exists": True}})

pipeline_recetas = [
    {"$match": {"atencion.recetas": {"$exists": True}}},
    {"$unwind": "$atencion.recetas"},
    {"$count": "total"}
]
res_recetas = list(db.citas.aggregate(pipeline_recetas))
recetas_mongo = res_recetas[0]["total"] if res_recetas else 0

# Tabla comparativa
evidencia_migracion = [
    {
        "Colección / Entidad": "centros_medicos",
        "MySQL COUNT(*)": get_mysql_count("centro_medico"),
        "MongoDB count_documents": db.centros_medicos.count_documents({}),
    },
    {
        "Colección / Entidad": "especialidades",
        "MySQL COUNT(*)": get_mysql_count("especialidad"),
        "MongoDB count_documents": db.especialidades.count_documents({}),
    },
    {
        "Colección / Entidad": "previsiones",
        "MySQL COUNT(*)": get_mysql_count("prevision"),
        "MongoDB count_documents": db.previsiones.count_documents({}),
    },
    {
        "Colección / Entidad": "medicos",
        "MySQL COUNT(*)": get_mysql_count("medico"),
        "MongoDB count_documents": db.medicos.count_documents({}),
    },
    {
        "Colección / Entidad": "pacientes",
        "MySQL COUNT(*)": get_mysql_count("paciente"),
        "MongoDB count_documents": db.pacientes.count_documents({}),
    },
    {
        "Colección / Entidad": "citas",
        "MySQL COUNT(*)": get_mysql_count("cita_medica"),
        "MongoDB count_documents": db.citas.count_documents({}),
    },
    {
        "Colección / Entidad": "atenciones (anidadas)",
        "MySQL COUNT(*)": get_mysql_count("atencion"),
        "MongoDB count_documents": atenciones_mongo,
    },
    {
        "Colección / Entidad": "recetas (anidadas)",
        "MySQL COUNT(*)": get_mysql_count("receta"),
        "MongoDB count_documents": recetas_mongo,
    }
]

cursor.close()
mysql_conn.close()

df_migracion = pd.DataFrame(evidencia_migracion)
df_migracion["Diferencia"] = df_migracion["MySQL COUNT(*)"] - df_migracion["MongoDB count_documents"]
df_migracion["Estado"] = df_migracion["Diferencia"].apply(lambda d: "OK (1:1)" if d == 0 else "ERROR")

display(df_migracion)

,Colección / Entidad,MySQL COUNT(*),MongoDB count_documents,Diferencia,Estado
0,centros_medicos,4,4,0,OK (1:1)
1,especialidades,8,8,0,OK (1:1)
2,previsiones,3,3,0,OK (1:1)
3,medicos,16,16,0,OK (1:1)
4,pacientes,42,42,0,OK (1:1)
5,citas,131,131,0,OK (1:1)
6,atenciones (anidadas),64,64,0,OK (1:1)
7,recetas (anidadas),86,86,0,OK (1:1)


In [39]:
from pymongo import MongoClient
from pprint import pprint

URI_REPLICA = (
    "mongodb://mongo1:30001,mongo2:30002,mongo3:30003/"
    "?replicaSet=my-replica-set&readPreference=primary&ssl=false"
)

client = MongoClient(URI_REPLICA)
db = client["salud_usm"]

print("Conexión establecida con el clúster:", client.topology_description)
print("Colecciones disponibles:", db.list_collection_names())

Conexión establecida con el clúster: <TopologyDescription id: 6ae8c6a7a37bdb0f250582bf, topology_type: ReplicaSetWithPrimary, servers: [<ServerDescription ('mongo1', 30001) server_type: RSPrimary, rtt: 0.002557121006248053>, <ServerDescription ('mongo2', 30002) server_type: Unknown, rtt: None>, <ServerDescription ('mongo3', 30003) server_type: Unknown, rtt: None>]>
Colecciones disponibles: ['pacientes', 'citas', 'medicos', 'previsiones', 'centros_medicos', 'especialidades']


### 4.3.a Agenda diaria de un médico

**Requerimiento:**
Dado el RUT de un médico y una fecha, devolver sus citas de ese día ordenadas por hora, mostrando hora, nombre del paciente, previsión, centro, especialidad y estado.

**Parámetros de prueba:**
* **RUT Médico:** `'22222222-2'` (Dr. Juan Pérez)
* **Fecha:** `'2026-10-15'`

**Explicación de la solución MQL:**
1. `$match`: Filtra los documentos de la colección `citas` que coincidan exactamente con el `medico.rut` y la `fecha` especificados.
2. `$sort`: Ordena cronológicamente los resultados de manera ascendente (`1`) según el campo `hora`.
3. `$project`: Proyecta únicamente los atributos requeridos (`hora`, `especialidad`, `estado`) y extrae los campos embebidos (`paciente.nombre`, `paciente.prevision`, `centro.nombre`), suprimiendo el identificador por defecto `_id: 0`.

**Consulta SQL equivalente (MySQL):**
```sql
SELECT c.hora,
       p.nombre_completo AS paciente,
       p.nombre_prevision AS prevision,
       cm.nombre_centro AS centro,
       c.nombre_especialidad AS especialidad,
       c.estado
FROM cita_medica c
JOIN paciente p ON c.rut_paciente = p.rut_paciente
JOIN centro_medico cm ON c.codigo_centro = cm.codigo_centro
WHERE c.rut_medico = '22222222-2'
  AND c.fecha = '2026-10-15'
ORDER BY c.hora ASC;

In [40]:
pipeline_a = [
    {
        "$match": {
            "medico.rut": "22222222-2",
            "fecha": "2026-10-15"
        }
    },
    {
        "$sort": { "hora": 1 }
    },
    {
        "$project": {
            "_id": 0,
            "hora": 1,
            "paciente": "$paciente.nombre",
            "prevision": "$paciente.prevision",
            "centro": "$centro.nombre",
            "especialidad": 1,
            "estado": 1
        }
    }
]

res_a = list(db.citas.aggregate(pipeline_a))
pprint(res_a)

[{'centro': 'Centro Médico Casa Central',
  'especialidad': 'Medicina General',
  'estado': 'Reservada',
  'hora': '09:00:00',
  'paciente': 'Juan Silva',
  'prevision': 'Fonasa'},
 {'centro': 'Centro Médico Casa Central',
  'especialidad': 'Medicina General',
  'estado': 'Confirmada',
  'hora': '09:30:00',
  'paciente': 'Camila Soto',
  'prevision': 'Fonasa'},
 {'centro': 'Centro Médico Casa Central',
  'especialidad': 'Cardiología',
  'estado': 'Confirmada',
  'hora': '10:00:00',
  'paciente': 'Pedro Morales',
  'prevision': 'Isapre'},
 {'centro': 'Centro Médico Casa Central',
  'especialidad': 'Medicina General',
  'estado': 'Reservada',
  'hora': '10:30:00',
  'paciente': 'Francisca Araya',
  'prevision': 'Particular'},
 {'centro': 'Centro Médico Casa Central',
  'especialidad': 'Cardiología',
  'estado': 'Confirmada',
  'hora': '11:00:00',
  'paciente': 'Andrés Castro',
  'prevision': 'Fonasa'}]


### 4.3.b Historial clínico de un paciente

**Requerimiento:**
Dado el RUT de un paciente, devolver todas sus atenciones ordenadas de la más reciente a la más antigua, mostrando fecha, centro, especialidad, nombre del médico, motivo de consulta, diagnósticos (código CIE-10 y descripción) y recetas (medicamento, dosis y días de tratamiento).

**Parámetros de prueba:**
* **RUT Paciente:** `'33333333-3'` (Juan Silva)

**Explicación de la solución MQL:**
1. `$match`: Selecciona únicamente las citas del paciente cuyo estado sea `'Atendida'` y que contengan el documento embebido `atencion` (`{ "$exists": True }`).
2. `$sort`: Aplica ordenamiento descendente (`-1`) sobre `fecha` y `hora` para presentar las consultas clínicas desde la más reciente a la más antigua.
3. `$project`: Reestructura la salida omitiendo `_id`, extrayendo la metadata del centro, especialidad y médico, y proyectando directamente los arreglos anidados `diagnosticos` y `recetas` sin requerir descomposiciones ni uniones costosas.

**Consulta SQL equivalente (MySQL):**
```sql
SELECT c.fecha,
       cm.nombre_centro AS centro,
       c.nombre_especialidad AS especialidad,
       m.nombre_completo AS medico,
       a.motivo_consulta,
       d.codigo_cie10,
       d.descripcion,
       r.medicamento,
       r.dosis,
       r.dias_tratamiento
FROM atencion a
JOIN cita_medica c ON a.id_cita = c.id_cita
JOIN centro_medico cm ON c.codigo_centro = cm.codigo_centro
JOIN medico m ON c.rut_medico = m.rut_medico
LEFT JOIN atencion_diagnostico ad ON a.id_atencion = ad.id_atencion
LEFT JOIN diagnostico d ON ad.codigo_cie10 = d.codigo_cie10
LEFT JOIN receta r ON a.id_atencion = r.id_atencion
WHERE c.rut_paciente = '33333333-3'
  AND c.estado = 'Atendida'
ORDER BY c.fecha DESC, c.hora DESC;

In [21]:
pipeline_b = [
    {
        "$match": {
            "paciente.rut": "33333333-3",
            "estado": "Atendida",
            "atencion": { "$exists": True }
        }
    },
    {
        "$sort": { "fecha": -1, "hora": -1 }
    },
    {
        "$project": {
            "_id": 0,
            "fecha": 1,
            "centro": "$centro.nombre",
            "especialidad": 1,
            "medico": "$medico.nombre",
            "motivo_consulta": "$atencion.motivo_consulta",
            "diagnosticos": "$atencion.diagnosticos",
            "recetas": "$atencion.recetas"
        }
    }
]

res_b = list(db.citas.aggregate(pipeline_b))
pprint(res_b)

[{'centro': 'Centro Médico Viña del Mar',
  'diagnosticos': [{'codigo_cie10': 'M54',
                    'descripcion': 'Dorsalgia / Lumbalgia'}],
  'especialidad': 'Traumatología',
  'fecha': '2026-09-30',
  'medico': 'Dr. Jorge Valenzuela',
  'motivo_consulta': 'Dolor cervical agudo',
  'recetas': [{'dias_tratamiento': 7,
               'dosis': 'Aplicar masaje suave cada 8 horas',
               'id_receta': 72,
               'medicamento': 'Ketoprofeno Gel 2.5%'}]},
 {'centro': 'Centro Médico Casa Central',
  'diagnosticos': [{'codigo_cie10': 'H10',
                    'descripcion': 'Conjuntivitis aguda'}],
  'especialidad': 'Oftalmología',
  'fecha': '2026-09-04',
  'medico': 'Dra. Valentina Castro',
  'motivo_consulta': 'Ardor ocular y enrojecimiento',
  'recetas': [{'dias_tratamiento': 7,
               'dosis': '1 gota cada 6 horas en ojo afectado',
               'id_receta': 32,
               'medicamento': 'Ciprofloxacino Oftálmico'}]},
 {'centro': 'Centro Médico Casa Cen

### 4.3.c Tasa de inasistencia por centro

**Requerimiento:**
Para cada centro médico, mostrar el total de citas, la cantidad de citas en estado 'No Asistió' y el porcentaje que representan (redondeado a 2 decimales), listando únicamente los centros cuyo porcentaje sea mayor a 15% y ordenando de mayor a menor porcentaje.

**Explicación de la solución MQL:**
1. `$group`: Agrupa por `centro.nombre`, calculando `total_citas` mediante un acumulador `$sum: 1` y `no_asistio` mediante un `$sum` condicional con `$cond` que evalúa si el estado es `'No Asistió'`.
2. `$project`: Calcula el porcentaje dividiendo citas no asistidas por el total, multiplicando por 100 y aplicando la función matemática `$round` a 2 posiciones decimales.
3. `$match`: Filtra únicamente los documentos agrupados cuyo `porcentaje_inasistencia` supere el umbral estipulado (`> 15.0`).
4. `$sort`: Ordena los centros resultantes de forma descendente (`-1`) por su tasa de inasistencia.

**Consulta SQL equivalente (MySQL):**
```sql
SELECT cm.nombre_centro AS centro,
       COUNT(*) AS total_citas,
       SUM(CASE WHEN c.estado = 'No Asistió' THEN 1 ELSE 0 END) AS no_asistio,
       ROUND((SUM(CASE WHEN c.estado = 'No Asistió' THEN 1 ELSE 0 END) / COUNT(*)) * 100, 2) AS porcentaje_inasistencia
FROM cita_medica c
JOIN centro_medico cm ON c.codigo_centro = cm.codigo_centro
GROUP BY cm.codigo_centro, cm.nombre_centro
HAVING porcentaje_inasistencia > 15.00
ORDER BY porcentaje_inasistencia DESC;

In [41]:
pipeline_c = [
    {
        "$group": {
            "_id": "$centro.nombre",
            "total_citas": { "$sum": 1 },
            "no_asistio": {
                "$sum": {
                    "$cond": [{ "$eq": ["$estado", "No Asistió"] }, 1, 0]
                }
            }
        }
    },
    {
        "$project": {
            "_id": 0,
            "centro": "$_id",
            "total_citas": 1,
            "no_asistio": 1,
            "porcentaje_inasistencia": {
                "$round": [
                    { "$multiply": [{ "$divide": ["$no_asistio", "$total_citas"] }, 100] },
                    2
                ]
            }
        }
    },
    {
        "$match": {
            "porcentaje_inasistencia": { "$gt": 15.0 }
        }
    },
    {
        "$sort": { "porcentaje_inasistencia": -1 }
    }
]

res_c = list(db.citas.aggregate(pipeline_c))
pprint(res_c)

[{'centro': 'Centro Médico San Joaquín',
  'no_asistio': 7,
  'porcentaje_inasistencia': 22.58,
  'total_citas': 31},
 {'centro': 'Centro Médico Casa Central',
  'no_asistio': 11,
  'porcentaje_inasistencia': 18.97,
  'total_citas': 58}]


### 4.3.d Ranking de medicamentos por especialidad

**Requerimiento:**
Dada una especialidad (Pediatría), mostrar los 5 medicamentos más recetados en las atenciones de esa especialidad junto a la cantidad de recetas de cada uno.

**Supuesto de Desempate (Documentado para README):**
En caso de presentarse una misma cantidad de recetas entre dos o más fármacos, se aplica como criterio de desempate el orden alfabético ascendente según el nombre del medicamento.

**Parámetros de prueba:**
* **Especialidad:** `'Pediatría'`

**Explicación de la solución MQL:**
1. `$match`: Filtra las citas correspondientes a la especialidad `'Pediatría'` que tengan el subarreglo `atencion.recetas`.
2. `$unwind`: Descompone el arreglo de recetas embebido, generando un documento individual por cada receta emitida.
3. `$group`: Agrupa por el atributo `atencion.recetas.medicamento` y totaliza las ocurrencias acumulando `$sum: 1`.
4. `$project`: Modela la presentación asignando el identificador al campo `medicamento` y renombrando la métrica como `cantidad_recetas`.
5. `$sort`: Aplica ordenamiento primario descendente (`-1`) por `cantidad_recetas` y secundario ascendente (`1`) por `medicamento` (criterio de desempate).
6. `$limit`: Limita la respuesta a los primeros 5 documentos para constituir el ranking.

**Consulta SQL equivalente (MySQL):**
```sql
SELECT r.medicamento,
       COUNT(*) AS cantidad_recetas
FROM receta r
JOIN atencion a ON r.id_atencion = a.id_atencion
JOIN cita_medica c ON a.id_cita = c.id_cita
WHERE c.nombre_especialidad = 'Pediatría'
GROUP BY r.medicamento
ORDER BY cantidad_recetas DESC, r.medicamento ASC
LIMIT 5;

In [42]:
pipeline_d = [
    {
        "$match": {
            "especialidad": "Pediatría",
            "atencion.recetas": { "$exists": True }
        }
    },
    {
        "$unwind": "$atencion.recetas"
    },
    {
        "$group": {
            "_id": "$atencion.recetas.medicamento",
            "cantidad_recetas": { "$sum": 1 }
        }
    },
    {
        "$project": {
            "_id": 0,
            "medicamento": "$_id",
            "cantidad_recetas": 1
        }
    },
    {
        "$sort": { "cantidad_recetas": -1, "medicamento": 1 }
    },
    {
        "$limit": 5
    }
]

res_d = list(db.citas.aggregate(pipeline_d))
pprint(res_d)

[{'cantidad_recetas': 8, 'medicamento': 'Paracetamol Gotas'},
 {'cantidad_recetas': 5, 'medicamento': 'Ibuprofeno Suspensión'},
 {'cantidad_recetas': 4, 'medicamento': 'Amoxicilina Suspensión'},
 {'cantidad_recetas': 2, 'medicamento': 'Cetirizina Jarabe'},
 {'cantidad_recetas': 2, 'medicamento': 'Mupirocina Crema'}]


### 4.3.e Pacientes multi-especialidad

**Requerimiento:**
Obtener los pacientes que han sido atendidos (citas en estado 'Atendida') en 3 o más especialidades distintas, mostrando su nombre, la cantidad de especialidades y el listado de ellas, ordenados de mayor a menor cantidad.

**Explicación de la solución MQL:**
1. `$match`: Selecciona exclusivamente los registros con estado `'Atendida'`.
2. `$group`: Agrupa por `paciente.rut`, conservando el nombre del paciente con `$first` y consolidando un conjunto de valores únicos para las disciplinas médicas usando el acumulador `$addToSet: "$especialidad"`.
3. `$project`: Modela los atributos de salida extrayendo el listado de especialidades y computando el tamaño del arreglo mediante el operador `$size`.
4. `$match`: Aplica la restricción de cardinalidad filtrando aquellos pacientes con `cantidad_especialidades` mayor o igual a 3 (`$gte: 3`).
5. `$sort`: Ordena de mayor a menor (`-1`) según la cantidad de especialidades y alfabéticamente (`1`) por el nombre del paciente.

**Consulta SQL equivalente (MySQL):**
```sql
SELECT p.nombre_completo AS paciente,
       COUNT(DISTINCT c.nombre_especialidad) AS cantidad_especialidades,
       GROUP_CONCAT(DISTINCT c.nombre_especialidad ORDER BY c.nombre_especialidad SEPARATOR ', ') AS listado_especialidades
FROM cita_medica c
JOIN paciente p ON c.rut_paciente = p.rut_paciente
WHERE c.estado = 'Atendida'
GROUP BY p.rut_paciente, p.nombre_completo
HAVING COUNT(DISTINCT c.nombre_especialidad) >= 3
ORDER BY cantidad_especialidades DESC;

In [43]:
pipeline_e = [
    {
        "$match": {
            "estado": "Atendida"
        }
    },
    {
        "$group": {
            "_id": "$paciente.rut",
            "nombre_paciente": { "$first": "$paciente.nombre" },
            "especialidades": { "$addToSet": "$especialidad" }
        }
    },
    {
        "$project": {
            "_id": 0,
            "paciente": "$nombre_paciente",
            "listado_especialidades": "$especialidades",
            "cantidad_especialidades": { "$size": "$especialidades" }
        }
    },
    {
        "$match": {
            "cantidad_especialidades": { "$gte": 3 }
        }
    },
    {
        "$sort": { "cantidad_especialidades": -1, "paciente": 1 }
    }
]

res_e = list(db.citas.aggregate(pipeline_e))
pprint(res_e)

[{'cantidad_especialidades': 5,
  'listado_especialidades': ['Dermatología',
                             'Neurología',
                             'Medicina General',
                             'Oftalmología',
                             'Traumatología'],
  'paciente': 'Juan Silva'},
 {'cantidad_especialidades': 4,
  'listado_especialidades': ['Pediatría',
                             'Neurología',
                             'Oftalmología',
                             'Dermatología'],
  'paciente': 'Camila Soto'},
 {'cantidad_especialidades': 4,
  'listado_especialidades': ['Cardiología',
                             'Traumatología',
                             'Ginecología',
                             'Medicina General'],
  'paciente': 'Pedro Morales'}]


## 4.4 Replicación y Alta Disponibilidad

### 4.4.1 Identificación y Estado de los Nodos del Replica Set

**Objetivo:**
Consultar desde Python el estado del clúster `my-replica-set`, identificando dinámicamente cuál nodo opera como `PRIMARY` y cuáles como `SECONDARY`.

In [ ]:
import pandas as pd

status = client.admin.command("replSetGetStatus")

miembros_info = []
for m in status["members"]:
    miembros_info.append({
        "ID": m["_id"],
        "Nodo / Host": m["name"],
        "Rol Actual": m["stateStr"],
        "Salud": "OK (1.0)" if m["health"] == 1.0 else "FALLO",
        "Tiempo Activo (s)": m.get("uptime", 0)
    })

df_miembros = pd.DataFrame(miembros_info)
display(df_miembros)

,ID,Nodo / Host,Rol Actual,Salud,Tiempo Activo (s)
0,0,mongo1:30001,PRIMARY,OK (1.0),113
1,1,mongo2:30002,SECONDARY,OK (1.0),112
2,2,mongo3:30003,SECONDARY,OK (1.0),112


### 4.4.2 Consistencia de Datos con WriteConcern Majority

**Objetivo:**
Demostrar que tras ejecutar operaciones de escritura en el Primario bajo la garantía de replicación `WriteConcern(w="majority")`, los datos se mantienen consistentes e idénticos en los 3 nodos (`mongo1:30001`, `mongo2:30002`, `mongo3:30003`).

Se evalúan las 3 operaciones exigidas por la pauta:
1. **Inserción:** Creación de una nueva cita con estado `'Reservada'`.
2. **Actualización:** Modificación masiva de citas vencidas en estado `'Reservada'` o `'Confirmada'` a `'Cancelada'`.
3. **Eliminación:** Remoción de una línea de receta dentro del subarreglo embebido `atencion.recetas` (usando `$pull`).

Para cada operación se realiza una verificación directa e independiente en cada nodo utilizando `directConnection=True`.

In [ ]:
from pymongo import MongoClient, WriteConcern

citas_majority = db.citas.with_options(write_concern=WriteConcern(w="majority", wtimeout=5000))

# Lista de nodos
nodos_cluster = [
    ("mongo1", 30001),
    ("mongo2", 30002),
    ("mongo3", 30003)
]

def verificar_consistencia_nodos(filtro, descripcion_prueba, campo_proyectar=None):
    resultados = []
    for host, puerto in nodos_cluster:
        c_direct = MongoClient(f"mongodb://{host}:{puerto}/?directConnection=true")
        
        my_state = c_direct.admin.command("replSetGetStatus")["myState"]
        rol = "PRIMARY" if my_state == 1 else "SECONDARY"
        
        col = c_direct["salud_usm"]["citas"]
        conteo = col.count_documents(filtro)
        
        detalle_extra = None
        if campo_proyectar:
            doc = col.find_one(filtro)
            detalle_extra = doc.get(campo_proyectar) if doc else None
            
        resultados.append({
            "Nodo": f"{host}:{puerto}",
            "Rol": rol,
            "Documentos coincidentes": conteo,
            "Valor Verificado": detalle_extra if detalle_extra is not None else "-"
        })
        c_direct.close()
        
    print(f"\n>>> Verificación: {descripcion_prueba}")
    df_verif = pd.DataFrame(resultados)
    display(df_verif)

In [46]:
# 1. Inserción de nueva cita
nueva_cita = {
    "_id": 9999,
    "id_cita": 9999,
    "fecha": "2026-11-20",
    "hora": "15:00:00",
    "estado": "Reservada",
    "paciente": {
        "rut": "33333333-3",
        "nombre": "Juan Silva",
        "prevision": "Fonasa"
    },
    "medico": {
        "rut": "22222222-2",
        "nombre": "Dr. Juan Pérez"
    },
    "centro": {
        "codigo": 101,
        "nombre": "Centro Médico Casa Central",
        "region": "Valparaíso"
    },
    "especialidad": "Medicina General"
}

res_insert = citas_majority.insert_one(nueva_cita)
print(f"Cita insertada en Primario con _id: {res_insert.inserted_id}")

# Verificar consistencia en los 3 nodos
verificar_consistencia_nodos(
    filtro={"id_cita": 9999},
    descripcion_prueba="Existencia de cita nueva (id_cita: 9999)"
)

Cita insertada en Primario con _id: 9999

>>> Verificación: Existencia de cita nueva (id_cita: 9999)


,Nodo,Rol,Documentos coincidentes,Valor Verificado
0,mongo1:30001,PRIMARY,1,-
1,mongo2:30002,SECONDARY,1,-
2,mongo3:30003,SECONDARY,1,-


In [47]:
# 2. Cancelar citas vencidas (fecha anterior a la actual 2026-11-02)
fecha_limite = "2026-11-02"

filtro_vencidas = {
    "estado": { "$in": ["Reservada", "Confirmada"] },
    "fecha": { "$lt": fecha_limite }
}

res_update = citas_majority.update_many(
    filtro_vencidas,
    { "$set": { "estado": "Cancelada" } }
)
print(f"Total de citas vencidas actualizadas a 'Cancelada': {res_update.modified_count}")

verificar_consistencia_nodos(
    filtro={"estado": "Cancelada"},
    descripcion_prueba="Total de citas en estado 'Cancelada' en cada nodo"
)

Total de citas vencidas actualizadas a 'Cancelada': 16

>>> Verificación: Total de citas en estado 'Cancelada' en cada nodo


,Nodo,Rol,Documentos coincidentes,Valor Verificado
0,mongo1:30001,PRIMARY,16,-
1,mongo2:30002,SECONDARY,16,-
2,mongo3:30003,SECONDARY,16,-


In [48]:
# 3. Eliminar una línea de receta específica de una atención (receta id_receta: 1 de la cita id_cita: 6)
id_cita_receta = 6
id_receta_eliminar = 1

doc_pre = db.citas.find_one({"id_cita": id_cita_receta})
recetas_antes = len(doc_pre.get("atencion", {}).get("recetas", []))
print(f"Recetas antes de eliminar en cita {id_cita_receta}: {recetas_antes}")

res_delete_receta = citas_majority.update_one(
    {"id_cita": id_cita_receta},
    {"$pull": {"atencion.recetas": {"id_receta": id_receta_eliminar}}}
)
print(f"Receta {id_receta_eliminar} eliminada. Documentos modificados: {res_delete_receta.modified_count}")

verificar_consistencia_nodos(
    filtro={"id_cita": id_cita_receta, "atencion.recetas.id_receta": id_receta_eliminar},
    descripcion_prueba=f"Comprobación de eliminación de receta {id_receta_eliminar} (debe ser 0 en todos)"
)

Recetas antes de eliminar en cita 6: 2
Receta 1 eliminada. Documentos modificados: 1

>>> Verificación: Comprobación de eliminación de receta 1 (debe ser 0 en todos)


,Nodo,Rol,Documentos coincidentes,Valor Verificado
0,mongo1:30001,PRIMARY,0,-
1,mongo2:30002,SECONDARY,0,-
2,mongo3:30003,SECONDARY,0,-


### 4.4.3 Prueba de Alta Disponibilidad y Failover

**Objetivo:**
Comprobar la tolerancia a fallos ante la desconexión abrupta del nodo Primario. Se detendrá el contenedor del Primario mediante Docker, evidenciando:
1. Elección automática de un nuevo nodo Primario por parte del clúster.
2. Continuidad operativa del cliente PyMongo utilizando la misma cadena de conexión (ejecutando lectura y una nueva escritura).
3. Reintegración limpia del nodo apagado una vez levantado de nuevo, asumiendo el rol de Secundario.

In [49]:
status_failover = client.admin.command("replSetGetStatus")
primario_original = next(m["name"] for m in status_failover["members"] if m["stateStr"] == "PRIMARY")
print(f"El nodo Primario actual es: {primario_original}")
print("\n--> PASO A SEGUIR: En tu terminal ejecuta el siguiente comando para apagarlo:")
print(f"    docker stop {primario_original.split(':')[0]}")

El nodo Primario actual es: mongo1:30001

--> PASO A SEGUIR: En tu terminal ejecuta el siguiente comando para apagarlo:
    docker stop mongo1


In [ ]:
import time

print("Esperando ventana de convergencia y elección del Replica Set (5 segundos)...")
time.sleep(5)

# 1. Detectar el nuevo nodo primario
status_durante = client.admin.command("replSetGetStatus")
nuevo_primario = next((m["name"] for m in status_durante["members"] if m["stateStr"] == "PRIMARY"), "En elección...")
print(f"--> NUEVO LÍDER ELECTO AUTOMÁTICAMENTE: {nuevo_primario}")

# 2. Ejecutar la Consulta 4.3.a sin cambiar la cadena de conexión
res_a_failover = list(db.citas.aggregate(pipeline_a))
print(f"--> LECTURA EXITOSA: Consulta 4.3.a retornó {len(res_a_failover)} citas durante la caída.")

citas_majority.delete_one({"_id": 9998})

# 3. Insertar una nueva cita
cita_emergencia = {
    "_id": 9998,
    "id_cita": 9998,
    "fecha": "2026-11-25",
    "hora": "11:00:00",
    "estado": "Reservada",
    "paciente": {"rut": "33333334-1", "nombre": "Camila Soto", "prevision": "Fonasa"},
    "medico": {"rut": "22222222-2", "nombre": "Dr. Juan Pérez"},
    "centro": {"codigo": 101, "nombre": "Centro Médico Casa Central", "region": "Valparaíso"},
    "especialidad": "Medicina General"
}

res_insert_failover = citas_majority.insert_one(cita_emergencia)
print(f"--> ESCRITURA EXITOSA: Cita de emergencia creada con ID {res_insert_failover.inserted_id}")

print("\n--> PASO A SEGUIR: Vuelve a encender el contenedor en tu terminal:")
print(f"    docker start {primario_original.split(':')[0]}")

Esperando ventana de convergencia y elección del Replica Set (5 segundos)...
--> NUEVO LÍDER ELECTO AUTOMÁTICAMENTE: mongo2:30002
--> LECTURA EXITOSA: Consulta 4.3.a retornó 5 citas durante la caída.
--> ESCRITURA EXITOSA: Cita de emergencia creada con ID 9998

--> PASO A SEGUIR: Vuelve a encender el contenedor en tu terminal:
    docker start mongo1


In [ ]:
print("Esperando 5 segundos a que el nodo reconecte y sincronice...")
time.sleep(5)

status_reintegrado = client.admin.command("replSetGetStatus")
resumen_final = []
for m in status_reintegrado["members"]:
    resumen_final.append({
        "Nodo": m["name"],
        "Rol Actual": m["stateStr"],
        "Salud": "OK" if m["health"] == 1.0 else "DOWN"
    })

df_reintegracion = pd.DataFrame(resumen_final)
display(df_reintegracion)

cliente_reintegrado = MongoClient(f"mongodb://{primario_original}/?directConnection=true")
tiene_cita_emergencia = cliente_reintegrado["salud_usm"]["citas"].count_documents({"id_cita": 9998})
print(f"\nVerificación en {primario_original}: Cita escrita en su ausencia encontrada: {tiene_cita_emergencia == 1}")
cliente_reintegrado.close()

Esperando 5 segundos a que el nodo reconecte y sincronice...


,Nodo,Rol Actual,Salud
0,mongo1:30001,PRIMARY,OK
1,mongo2:30002,SECONDARY,OK
2,mongo3:30003,SECONDARY,OK



Verificación en mongo1:30001: Cita escrita en su ausencia encontrada: True
